# Reggression Model Training

To detect the features of the nodules we will train a regression model to predict the nodule features vector from a given image

## Imports
we need to first need to imports necessery packages

In [ ]:
import torch
import lightning as L
import numpy as np
import pandas as pd
from pathlib import Path

from models.regression_model import NoduleFeaturesModel
from data_modules.regression_dataset_module import RegressionDataModule

from constants.detection.model_constants import NoduleFeatures,RegressionModelConstants,Accelerator
from constants.detection.dataset_constants import DatasetConstants
from constants.common import Metrics,ModelStage

import lightning.pytorch.callbacks as LightningCallbacks
from utils.callbacks.notification_callback import NtfyCallback
from lightning.pytorch.loggers import TensorBoardLogger

## Model and dataset preparation

since we need 1:1 alignment for patients, all the info is stored in a csv containing info for both the reggresion model and detection model, we'll pass the path to the csv to the lightning data module and set up the model,data module and trainer

In [ ]:
reg_df_path = DatasetConstants.DATASET_DIR

DATASET_ROOT = DatasetConstants.PROJECT_ROOT / "DetecetionModel"

reg_dm = RegressionDataModule(
    metadata_csv=reg_df_path,
    dataset_root=DATASET_ROOT
)

model = NoduleFeaturesModel()

callbacks=[
    LightningCallbacks.EarlyStopping(
        monitor=Metrics.DEFAULT_LOSS.get_variant(ModelStage.VAL),
        patience=5,

    ),
    NtfyCallback(
        model_name=RegressionModelConstants.MODEL_NAME,
    ),
    LightningCallbacks.ModelCheckpoint(
        dirpath=RegressionModelConstants.CHECKPOINT_DIR,
        filename=RegressionModelConstants.BEST_MODEL_CHECKPOINT_NAME,
        mode="min",
        save_top_k=1
    ),
    LightningCallbacks.OnExceptionCheckpoint(
        dirpath=RegressionModelConstants.CHECKPOINT_DIR,
        filename=RegressionModelConstants.EXCEPTION_CHECKPOINT_FILE_NAME
    ),
    LightningCallbacks.RichProgressBar()
]

trainer= L.Trainer(accelerator=Accelerator.AUTO,
                   devices=1,
                   logger=TensorBoardLogger(save_dir=RegressionModelConstants.LOG_DIR,
                                            name=RegressionModelConstants.MODEL_NAME.replace(" ","_")))

